[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/milioe/casos-ia-ibero-diplomado/blob/main/Modulo%204%3A%20NLP/03-OCRfacturas.ipynb)


# 03 — De factura en imagen a texto: primero intentamos `pypdf`

## La situación

Tienes **facturas como imágenes** (JPEG en `batch_1`, dataset [High-Quality Invoice Images for OCR](https://www.kaggle.com/datasets/osamahosamabdellatif/high-quality-invoice-images-for-ocr)). Quieres pasarlas a **texto**.

En **`02-PDF_reporte`** usamos **`pypdf`** con un PDF que ya traía texto seleccionable. Aquí probamos si eso basta cuando el “documento” es en realidad una **foto**.

Este notebook queda preparado para **Google Colab** sin depender de `data/`: trabaja directo con `batch_1/` en la carpeta actual y, si solo tienes el ZIP, lo descomprime automáticamente.

---

## Requisitos

- En Colab, subir **`batch_1.zip`** (o tener ya la carpeta **`batch_1/`**) en la carpeta actual.
- Internet la primera vez que corre **EasyOCR** (descarga modelos).

In [ ]:
# %pip install -q pypdf img2pdf easyocr pillow pandas tqdm numpy matplotlib

## Descomprimir el ZIP (solo si aún no tienes `batch_1/`)

Si **ya** ves las carpetas `batch1_1`, etc. dentro de `batch_1/`, puedes **saltar** la celda de código siguiente.

In [ ]:
import zipfile
from pathlib import Path

# En Colab normalmente estamos en /content
BASE = Path.cwd().resolve()
CARPETA_FACTURAS = BASE / "batch_1"


def encontrar_batch_zip():
    """Busca batch_1.zip en carpeta actual y hasta 3 niveles arriba."""
    candidatos = [BASE, *list(BASE.parents)[:3]]
    for carpeta in candidatos:
        z = carpeta / "batch_1.zip"
        if z.is_file():
            return z
    return None


hay_jpgs = CARPETA_FACTURAS.exists() and any(CARPETA_FACTURAS.glob("**/*.jpg"))
if hay_jpgs:
    print("Listo: ya hay imágenes en", CARPETA_FACTURAS)
else:
    z = encontrar_batch_zip()
    if z is None:
        raise FileNotFoundError(
            "No encontré batch_1.zip. Súbelo a Colab (panel Files) o crea batch_1/ manualmente."
        )

    print("Descomprimiendo", z, "...")
    with zipfile.ZipFile(z, "r") as zf:
        zf.extractall(BASE)

    if not (CARPETA_FACTURAS.exists() and any(CARPETA_FACTURAS.glob("**/*.jpg"))):
        raise FileNotFoundError(
            "Se descomprimió el ZIP, pero no apareció batch_1/ con archivos .jpg."
        )

    print("Imágenes en:", CARPETA_FACTURAS)

## Rutas que usaremos

`CARPETA_FACTURAS` apunta a `batch_1/`, donde están las `.jpg`.

In [ ]:
from pathlib import Path

CARPETA_FACTURAS = Path.cwd().resolve() / "batch_1"

---

## Intento 1: `pypdf` (como en el notebook 02)

Vamos paso a paso: cargar una factura, meterla en un PDF con **`img2pdf`**, pedir el texto con **`pypdf`**, y **ver** imagen vs texto.

### Paso 1 — Cargar la primera factura (bytes del JPEG)

Ordenamos las rutas y tomamos la primera; `read_bytes()` lee el archivo completo en memoria.

In [ ]:
rutas_jpg = sorted(CARPETA_FACTURAS.glob("**/*.jpg"))
ruta_factura = rutas_jpg[0]

bytes_imagen = ruta_factura.read_bytes()

print(ruta_factura)
print(len(bytes_imagen), "bytes")

### Paso 2 — PDF de una página + `extract_text()`

`img2pdf` **no** lee letras: solo empaqueta la imagen dentro de un PDF.

In [ ]:
import io

import img2pdf
from pypdf import PdfReader

pdf_bytes = img2pdf.convert(bytes_imagen)
lector = PdfReader(io.BytesIO(pdf_bytes))
texto_con_pypdf = lector.pages[0].extract_text() or ""

### Paso 3 — Ver la factura como imagen

Una sola figura: izquierda la factura, derecha lo que devolvió `pypdf` (y una frase de contexto).

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

img = Image.open(io.BytesIO(bytes_imagen)).convert("RGB")

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

axes[0].imshow(img)
axes[0].set_title("Imagen de la factura")
axes[0].axis("off")

axes[1].axis("off")
axes[1].set_title("Texto que devolvió pypdf")
nota = (
    "pypdf solo lee texto ya guardado en el PDF.\n"
    "Aquí el PDF es solo la foto embebida.\n\n"
    f"Resultado: {repr(texto_con_pypdf)}"
)
axes[1].text(0.02, 0.95, nota, transform=axes[1].transAxes, va="top", fontsize=10, family="monospace")

plt.suptitle(str(ruta_factura), fontsize=9, y=1.02)
plt.tight_layout()
plt.show()

### Qué acabamos de ver

- **`pypdf`** no hace OCR: si no hay texto embebido, obtienes `''`.
- Para leer **píxeles → letras** hace falta un motor de **OCR**.

---

## Intento 2: OCR con EasyOCR

| Opción | Idea general |
|--------|----------------|
| **EasyOCR** | Instalación con pip; primera vez descarga modelos |
| **Tesseract** | Muy usado; requiere instalar el programa en el sistema |
| **Servicios en la nube** | Suele dar muy buena calidad; hay coste y cuenta |

Este lote de facturas va bien con idioma **`en`**.

In [ ]:
import json
import random
import re

import easyocr
import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

# Inglés (lote de facturas) y español (tickets / texto en ES). Tras cambiar la lista, vuelve a ejecutar la celda que crea `lector_ocr` para descargar/cargar el modelo.
IDIOMAS_OCR = ["es", "en"]
USAR_GPU = False

In [ ]:
lector_ocr = easyocr.Reader(IDIOMAS_OCR, gpu=USAR_GPU)

Tres celdas seguidas:

1. Elegir al azar una factura y obtener el texto con el OCR (variable `texto_ocr`).
2. Mostrar **solo la imagen** con matplotlib.
3. Imprimir el texto con **`print`**: así aparece como salida normal del notebook (**seleccionable y copiable**), no dibujado como gráfico.

Cada vez que vuelvas a ejecutar esas celdas, otra factura al azar. Más abajo, en **OCR en varias facturas**, el límite de cuántas procesar se declara en la misma celda del bucle (no hace falta definirlo antes).

In [ ]:
import numpy as np
from PIL import Image

todas = sorted(CARPETA_FACTURAS.glob("**/*.jpg"))
factura_al_azar = random.choice(todas)

img = Image.open(factura_al_azar).convert("RGB")
detecciones = lector_ocr.readtext(np.asarray(img), detail=1)
texto_ocr = "\n".join(t[1] for t in detecciones)

print("Archivo:", factura_al_azar)
print("Total en el lote:", len(todas), "archivos .jpg")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.patches import Polygon

fig, ax = plt.subplots(figsize=(10, 12))
ax.imshow(img)
for det in detecciones:
    ax.add_patch(
        Polygon(np.array(det[0]), closed=True, fill=False, edgecolor="lime", linewidth=1.2)
    )
ax.set_title("Factura elegida al azar — recuadros = zonas que detectó el OCR")
ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
print("— Texto reconocido por el OCR (salida literal; puedes copiarlo) —\n")
print(texto_ocr)

## OCR en varias facturas

Para cada imagen: OCR → texto largo → **regex** para llenar **muchas columnas** (cabecera, direcciones, impuestos, IBAN, resumen y totales). Los **renglones de producto** van en la columna **`items_json`**: es un **texto JSON** (lista de objetos) que puedes copiar o volver a leer con `json.loads` en Python; en Excel se ve como cadena, y eso está bien para clase.

Elegimos **`LIMITE_IMAGENES` facturas al azar** del lote (no las primeras en orden), para que cada ejecución mezcle ejemplos. Si quisieras siempre las mismas rutas, podrías fijar `random.seed(123)` justo antes del `sample`.

In [ ]:
def imagen_a_texto(lector, img_pil):
    # EasyOCR no entiende el archivo en disco tal cual: prefiere un arreglo de píxeles (numpy).
    # convertimos la imagen PIL a ese formato y se la pasamos al modelo ya cargado en `lector`.
    arr = np.asarray(img_pil)
    # readtext devuelve una lista de detecciones; con detail=1 cada elemento trae cajas, texto y confianza.
    # a nosotros solo nos importa el string reconocido (índice 1), que es la "franja" de texto que vio.
    detecciones = lector.readtext(arr, detail=1)
    # juntamos todas esas tiras en un solo bloque, línea por línea, como si fuera un .txt pegado.
    return "\n".join(t[1] for t in detecciones)


def _es_encabezado_tabla_items(ln: str) -> bool:
    return ln.lower() in {
        "no.",
        "description",
        "qty",
        "um",
        "net price",
        "net worth",
        "vat [%]",
        "vat",
        "gross",
        "worth",
        "gross worth",
        "items",
    }


def _termina_bloque_descripcion_item(i: int, lineas: list) -> bool:
    # en esta plantilla, la descripción acaba cuando viene la cantidad tipo "1,00" y en la siguiente línea la unidad ("each", etc.).
    if i >= len(lineas):
        return False
    ln = lineas[i].strip()
    if not re.match(r"^\d+[.,]\d{2}$", ln):
        return False
    if i + 1 < len(lineas):
        nxt = lineas[i + 1].lower().strip()
        if nxt in ("each", "pc", "pcs", "unit", "hr", "kg", "lb", "box", "set", "lot"):
            return True
    return False


def extraer_items_ocr(texto: str) -> list:
    """Devuelve una lista de dicts (una entrada por renglón de producto). El OCR a veces junta o parte líneas;
    esto es una heurística razonable para la plantilla del dataset, no garantía al 100 %."""
    t = texto.replace("\r\n", "\n")
    if not re.search(r"(?i)ITEMS", t):
        return []
    _, tail = re.split(r"(?i)ITEMS", t, maxsplit=1)
    part = re.split(r"(?i)\bSUMMARY\b", tail, maxsplit=1)[0]
    lineas = [ln.strip() for ln in part.splitlines() if ln.strip()]
    items = []
    i = 0
    while i < len(lineas) and _es_encabezado_tabla_items(lineas[i]):
        i += 1
    orden = [
        "cantidad",
        "unidad_medida",
        "precio_neto",
        "importe_neto_linea",
        "iva_porcentaje",
        "importe_bruto_linea",
    ]
    while i < len(lineas):
        while i < len(lineas) and (
            _es_encabezado_tabla_items(lineas[i])
            or (len(lineas[i]) < 3 and lineas[i].replace(".", "").isdigit())
        ):
            i += 1
        if i >= len(lineas):
            break
        desc_parts = []
        while i < len(lineas):
            if _termina_bloque_descripcion_item(i, lineas) and desc_parts:
                break
            ln = lineas[i]
            if _es_encabezado_tabla_items(ln):
                break
            if not desc_parts and (re.fullmatch(r"[\d\s,\.%\$]+", ln) and len(ln) < 8):
                i += 1
                continue
            desc_parts.append(ln)
            i += 1
        if not desc_parts:
            if i < len(lineas):
                i += 1
            continue
        descripcion = " ".join(desc_parts)
        item = {k: "" for k in ["descripcion", *orden]}
        item["descripcion"] = descripcion[:500]
        j = 0
        while i < len(lineas) and j < len(orden):
            ln = lineas[i]
            if _es_encabezado_tabla_items(ln):
                break
            if (
                j == 0
                and len(ln) > 15
                and re.search(r"[A-Za-z]{5,}", ln)
                and not re.match(r"^\d+[.,]\d{2}$", ln.strip())
            ):
                break
            item[orden[j]] = ln
            j += 1
            i += 1
        items.append(item)
    return items


def extraer_campos_factura_ocr(texto: str) -> dict:
    """Del texto plano del OCR armamos muchas columnas. Los productos van aparte y al final los volcamos a JSON."""
    t = texto.replace("\r\n", "\n")
    vacio = {
        "no_factura": "",
        "fecha_emision": "",
        "vendedor": "",
        "cliente": "",
        "direccion_calle_vendedor": "",
        "direccion_ciudad_vendedor": "",
        "direccion_calle_cliente": "",
        "direccion_ciudad_cliente": "",
        "tax_id_vendedor": "",
        "tax_id_cliente": "",
        "iban": "",
        "num_items": 0,
        "items_json": "[]",
        "resumen_iva_porcentaje": "",
        "resumen_neto": "",
        "resumen_iva_monto": "",
        "resumen_bruto": "",
        "total": "",
        "primer_concepto": "",
    }
    out = dict(vacio)

    m = re.search(r"(?i)(?:invoice\s*)?no:\s*([A-Za-z0-9\-]+)", t)
    if m:
        out["no_factura"] = m.group(1).strip()

    m = re.search(r"(?i)date\s+of\s+issue:\s*([\d./\-\s]+)", t)
    if m:
        out["fecha_emision"] = re.sub(r"\s+", " ", m.group(1)).strip()

    m = re.search(
        r"(?i)Seller:\s*\n\s*Client:\s*\n\s*([^\n]+)\s*\n\s*([^\n]+)\s*\n\s*([^\n]+)\s*\n\s*([^\n]+)\s*\n\s*([^\n]+)\s*\n\s*([^\n]+)",
        t,
    )
    if m:
        out["vendedor"] = m.group(1).strip()
        out["cliente"] = m.group(2).strip()
        out["direccion_calle_vendedor"] = m.group(3).strip()
        out["direccion_calle_cliente"] = m.group(4).strip()
        out["direccion_ciudad_vendedor"] = m.group(5).strip()
        out["direccion_ciudad_cliente"] = m.group(6).strip()
    else:
        m = re.search(r"(?i)Seller:\s*\n\s*Client:\s*\n\s*([^\n]+)\s*\n\s*([^\n]+)", t)
        if m:
            out["vendedor"] = m.group(1).strip()
            out["cliente"] = m.group(2).strip()
        else:
            m = re.search(r"(?i)Client:\s*\n+\s*([^\n]+)", t)
            if m:
                out["cliente"] = m.group(1).strip()

    ids = re.findall(r"(?i)Tax Id:\s*([0-9\-]+)", t)
    if len(ids) >= 1:
        out["tax_id_vendedor"] = ids[0].strip()
    if len(ids) >= 2:
        out["tax_id_cliente"] = ids[1].strip()

    m = re.search(r"(?i)IBAN:\s*(\S+)", t)
    if m:
        out["iban"] = re.sub(r"[^\w]+$", "", m.group(1).strip())

    items = extraer_items_ocr(t)
    out["num_items"] = len(items)
    out["items_json"] = json.dumps(items, ensure_ascii=False)
    if items:
        out["primer_concepto"] = items[0]["descripcion"][:200]

    m = re.search(r"(?i)SUMMARY\s*\n(.*?)(?=\n\s*Total\s*\n)", t, re.DOTALL)
    if m:
        bloque = m.group(1)
        pcts, nums = [], []
        for ln in bloque.splitlines():
            ln = ln.strip()
            if not ln:
                continue
            if re.match(r"^\d+\s*%$", ln):
                pcts.append(ln)
            elif re.match(r"^[\d,\.]+$", ln):
                nums.append(ln)
        if pcts:
            out["resumen_iva_porcentaje"] = pcts[-1]
        if len(nums) >= 3:
            out["resumen_neto"] = nums[0]
            out["resumen_iva_monto"] = nums[1]
            out["resumen_bruto"] = nums[2]

    m = re.search(r"(?i)Total\s*\n((?:\s*\$?\s*[\d,\.]+\s*\n)+)", t)
    if m:
        nums = re.findall(r"\$?\s*([\d,\.]+)", m.group(1))
        if nums:
            out["total"] = nums[-1].strip()

    if not out["primer_concepto"] and re.search(r"(?i)ITEMS", t):
        _, resto = re.split(r"(?i)ITEMS", t, maxsplit=1)
        lineas = [ln.strip() for ln in resto.splitlines() if ln.strip()]
        for ln in lineas:
            if _es_encabezado_tabla_items(ln) or len(ln) < 4:
                continue
            if re.fullmatch(r"[\d\s,\.%\$]+", ln):
                continue
            out["primer_concepto"] = ln[:200]
            break

    return out


# cuántas facturas tocamos en la demo; elige rutas al azar (sin repetir dentro de esta corrida).
LIMITE_IMAGENES = 5
k = min(LIMITE_IMAGENES, len(todas))
rutas_para_ocr = random.sample(todas, k=k)
print(
    f"OCR sobre {k} imagen(es) elegidas al azar de {len(todas)} en el lote "
    f"(LIMITE_IMAGENES = {LIMITE_IMAGENES})."
)
for p in rutas_para_ocr:
    print(" ·", p)

filas = []
for ruta in tqdm(rutas_para_ocr, desc="OCR"):
    img_pil = Image.open(ruta).convert("RGB")
    texto_completo = imagen_a_texto(lector_ocr, img_pil)
    campos = extraer_campos_factura_ocr(texto_completo)
    filas.append({"archivo": str(ruta), "texto_completo": texto_completo, **campos})

df_ocr = pd.DataFrame(filas)

# todas las columnas “estructuradas”; texto_completo e items_json son largos (pandas los recorta al mostrar).
pd.set_option("display.max_colwidth", 72)
columnas_resumen = [
    "archivo",
    "no_factura",
    "fecha_emision",
    "vendedor",
    "cliente",
    "direccion_calle_vendedor",
    "direccion_ciudad_vendedor",
    "direccion_calle_cliente",
    "direccion_ciudad_cliente",
    "tax_id_vendedor",
    "tax_id_cliente",
    "iban",
    "num_items",
    "items_json",
    "resumen_iva_porcentaje",
    "resumen_neto",
    "resumen_iva_monto",
    "resumen_bruto",
    "total",
    "primer_concepto",
]
df_ocr[columnas_resumen]

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.patches import Polygon

NUMERO = ""  # ej. "47703225"
ruta = df_ocr.loc[df_ocr["texto_completo"].str.contains(NUMERO, na=False), "archivo"].iloc[0]
img = Image.open(ruta).convert("RGB")
d = lector_ocr.readtext(np.asarray(img), detail=1)

fig, ax = plt.subplots(figsize=(10, 12))
ax.imshow(img)
for det in d:
    ax.add_patch(Polygon(np.array(det[0]), closed=True, fill=False, edgecolor="lime", linewidth=1.2))
ax.set_title("Misma factura — cuadros del OCR")
ax.axis("off")
plt.tight_layout()
plt.show()

print("\n".join(x[1] for x in d))


## Otro archivo: `Ticket.jpeg`

Mismo **`lector_ocr`** (EasyOCR con los idiomas que definiste arriba). Imagen + **cajas** de detección y abajo el **texto** unido, para comparar con las facturas del lote.

**Fotos enormes (12 MP+):** reducir el lado largo a ~**2000 px** suele arreglar el “spam” de `8` y cajas absurdas. **Con EXIF del móvil,** `exif_transpose` **ya** aplica el giro; no sumes 90° extra salvo que el ticket **siga** de costado. Puedes activar en código **`PROBAR_4_ANGULOS`** para probar 0/90/180/270° y quedarse con la mejor puntuación (tarda más).

**Idioma:** conviene `IDIOMAS_OCR` con **`"es"`** (además de `"en"`) y volver a crear el lector, si el ticket está en español.

En Colab, deja `Ticket.jpeg` en la carpeta actual (o ajusta la ruta en la celda).

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.patches import Polygon
from PIL import Image, ImageEnhance, ImageOps

ruta_ticket = Path.cwd().resolve() / "Ticket.jpeg"
img = Image.open(ruta_ticket).convert("RGB")
# Giro que guarda el móvil (p. ej. EXIF 6). Sin esto el texto en píxeles queda de lado.
img = ImageOps.exif_transpose(img)
# Cuidado: si ya aplicas exif_transpose, añadir 90° vuelve a poner el ticket "mal" y el OCR otra vez falla.
# Úsalo solo si, viendo la imagen, el texto aún se lee en vertical. Si no, deja 0.
ROTACION_GRADOS = 0
if ROTACION_GRADOS:
    img = img.rotate(ROTACION_GRADOS, expand=True, resample=Image.BICUBIC, fillcolor="white")

# 6000+ px: EasyOCR a menudo degrada a cajas basura. ~2000 px de lado es un buen balance.
MAX_LADO = 2000
w, h = img.size
if max(w, h) > MAX_LADO:
    s = MAX_LADO / max(w, h)
    img = img.resize((max(1, int(w * s)), max(1, int(h * s))), Image.LANCZOS)

img = ImageEnhance.Contrast(img).enhance(1.2)

# True = prueba 0, 90, 180, 270° y se queda con la que más confianza suma (cuesta ~4x el tiempo de OCR).
PROBAR_4_ANGULOS = True

if PROBAR_4_ANGULOS:
    mejor = None  # (tupla_puntuación, detecciones, imagen, grados)
    for ang in (0, 90, 180, 270):
        t = img if ang == 0 else img.rotate(ang, expand=True, resample=Image.BICUBIC, fillcolor="white")
        d_try = lector_ocr.readtext(np.asarray(t), detail=1)
        confs = sum(x[2] for x in d_try) if d_try else 0.0
        n = len("".join(x[1] for x in d_try)) if d_try else 0
        clave = (confs, n)
        if mejor is None or clave > mejor[0]:
            mejor = (clave, d_try, t, ang)
    clave, d, img, ang_mejor = mejor
    print(
        f"OCR: orientación elegida = {ang_mejor}° (suma conf. ≈ {clave[0]:.1f}, {clave[1]} caracteres; desempata por nº de caracteres)\n"
    )
else:
    d = lector_ocr.readtext(np.asarray(img), detail=1)
    ang_mejor = None  # no se probó giro múltiple

fig, ax = plt.subplots(figsize=(12, 14))
ax.imshow(img)
for det in d:
    ax.add_patch(
        Polygon(np.array(det[0]), closed=True, fill=False, edgecolor="lime", linewidth=1.2)
    )
suf = f" + giro {ang_mejor}°" if (PROBAR_4_ANGULOS and ang_mejor is not None) else ""
ax.set_title(f"{ruta_ticket} — EasyOCR (EXIF + {MAX_LADO}px + contraste{suf})")
ax.axis("off")
plt.tight_layout()
plt.show()

print("— Texto OCR —\n")
print("\n".join(x[1] for x in d))
